In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.optimize import fmin_slsqp
from toolz import reduce, partial
import pyfixest as pf
from pyfixest import iplot

In [8]:
aqi_data = pd.read_csv("aqi_daily_1980_to_2021.csv")

In [19]:
rb_states = [
    "Illinois",
    "Indiana",
    "Michigan",
    "New York",
    "Ohio",
    "Pennsylvania",
    "West Virginia",
    "Wisconsin"
]

res_counties = ["Lake", "Marion", "Kent", "Macomb", "Wayne", "Monroe", "Hamilton", "Montgomery"]

rb_aqi_data = aqi_data[aqi_data["State Name"].isin(rb_states) & aqi_data["County Name"].isin(res_counties)]

rb_aqi_data["Date"] = pd.to_datetime(rb_aqi_data["Date"])
rb_aqi_data["year"] = rb_aqi_data["Date"].dt.year

rb_aqi_summary = (rb_aqi_data.groupby(["County Name", "Defining Parameter", "year"], as_index=False)["AQI"]
                  .mean().rename(columns={"AQI": "avg_aqi"}))

rb_aqi_summary = rb_aqi_summary[(rb_aqi_summary["year"] >= 2000) & (rb_aqi_summary["year"] <= 2010)]
rb_aqi_summary
#rb_aqi_summary.to_csv("aqi_summary.csv", index=False)

/var/folders/g0/1_gj42z94q93wf4k_ck_hqd40000gn/T/ipykernel_35442/2403439424.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rb_aqi_data["Date"] = pd.to_datetime(rb_aqi_data["Date"])
/var/folders/g0/1_gj42z94q93wf4k_ck_hqd40000gn/T/ipykernel_35442/2403439424.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rb_aqi_data["year"] = rb_aqi_data["Date"].dt.year


,County Name,Defining Parameter,year,avg_aqi
16,Hamilton,CO,2006,33.000000
39,Hamilton,NO2,2000,41.090909
40,Hamilton,NO2,2001,40.529412
41,Hamilton,NO2,2002,38.062500
42,Hamilton,NO2,2003,44.448276
...,...,...,...,...
1384,Wayne,SO2,2006,80.397436
1385,Wayne,SO2,2007,79.148148
1386,Wayne,SO2,2008,77.246377
1387,Wayne,SO2,2009,67.681818


In [20]:
rb_aqi_summary["treated"] = (rb_aqi_summary["County Name"] == "Wayne").astype(int)

rb_aqi_wide = (
    rb_aqi_summary
      .pivot_table(
          index=["County Name", "year"],     
          columns="Defining Parameter",
          values="avg_aqi",
          aggfunc="mean"
      )
      .reset_index()
)
rb_aqi_wide.columns.name = None

rb_aqi_wide["treated"] = (rb_aqi_wide["County Name"] == "Wayne").astype(int)




In [21]:
rb_aqi_wide

,County Name,year,CO,NO2,Ozone,PM10,PM2.5,SO2,treated
0,Hamilton,2000,NaN,41.090909,48.050847,NaN,68.502262,90.750000,0
1,Hamilton,2001,NaN,40.529412,52.940547,NaN,66.582011,98.904110,0
2,Hamilton,2002,NaN,38.062500,59.468750,NaN,60.465608,89.337349,0
3,Hamilton,2003,NaN,44.448276,54.018703,NaN,61.238318,78.043478,0
4,Hamilton,2004,NaN,43.225806,48.139334,NaN,60.497110,83.181818,0
...,...,...,...,...,...,...,...,...,...
83,Wayne,2006,NaN,31.227273,41.375862,41.181818,65.434286,80.397436,1
84,Wayne,2007,NaN,33.352941,53.932075,47.750000,59.970732,79.148148,1
85,Wayne,2008,NaN,27.925926,44.460526,29.142857,55.994536,77.246377,1
86,Wayne,2009,NaN,31.555556,39.028125,41.000000,52.792929,67.681818,1


In [22]:
pollutants = [c for c in rb_aqi_wide.columns if c not in ["County Name", "year", "treated"]]

available_all_counties_all_years = [
    c for c in pollutants
    if rb_aqi_wide[c].notna().all()
]

available_all_counties_all_years


['Ozone', 'PM2.5']